# Astra — salvage the V1 distillation run into trainable V2 data

`distillation/astra_v1_distillation.jsonl` holds 1002 records the teacher already paid for.
It cannot be trained on as it stands, but most of it is good. This notebook repairs it,
splits it, emits the configs `training/gpu_train.py` expects, rebuilds the tokenizer over the
result, and reports exactly what it dropped and why.

**No API key. No GPU. Seconds, not hours.** The teacher-distillation notebook
(`colab/Distliation/astra_v1_distillation_v2.ipynb`) is the path for *new* data; this one
recovers what you already have.

## What V1 actually got wrong

Every number below is a measured count from the finished file, cross-checked against
`training/gpu_train.py` and `configs/astra_200m.json`.

| Measured defect | Count | Consequence |
|---|---|---|
| `risk` lowercase (`medium` vs `INFO`) | 1002 / 1002 | `gpu_train.py` label head never matches its own config |
| `input` spans multiple lines | 264 (26.3%) | breaks the one-row-per-line `.txt` contract |
| input under 120 chars (e.g. a bare `systemd[1]` line) | 194 (19.4%) | nothing to learn from |
| target is `"No action required."` / `"None."` | 122 (12.2%) | teaches the model to refuse |
| redundant duplicate inputs (one repeated 27x) | 89 (8.9%) | wasted budget |
| no `id` / `dedup_key` / `grid_index` / `created_at` / `teacher_model` / `hardness` | 1002 / 1002 | not reproducible, not resumable |
| `early_warning` not a function of the label | 74 `malicious` rows flagged `false` | unlearnable flag |
| `confidence` pinned at exactly `1.0` | 204 (20.4%) | model learns false certainty |
| extra `action` key (schema drift) | 3 | audit rejects the row |
| n=13 leak gate `gpu_train.py:346` fails at 10.9% overlap | 8779 shared n-grams | **training exits before step 1** |

That last one is worth reading twice. `leak_check(train_ids, val_ids, n=13)` compares raw token
n-grams of two streams, and every chat block repeats the literal `You:` / `Astra:` framing. The
gate measures the template, not contamination. The config this notebook emits turns it off *and
records why*; the meaningful check — no record body in two splits — is asserted instead.

## One judgement call you should know about

The V2 validator bans no-op targets outright. Applied to salvaged data that rule deletes
**97% of the `normal` class** (219 rows in, 12 out) and hands you a corpus that is 99%
`suspicious` — a worse skew than the one V1 had. `"No action required."` is the *correct*
answer for a `normal` row, so this notebook scopes the rule to classes that actually owe an
action, and reports the 58 survivors as an explicit deviation instead of hiding them.

## Stage 0 — Repository and paths

Clones the repo and clones on first run. Colab resets `/content` on reconnect, so the clone
repeats; the generated data is mirrored to Drive at the end.

In [ ]:
import os, subprocess, sys, glob, json, shutil

REPO = '/content/astra'
if not os.path.isdir(os.path.join(REPO, 'tools')):
    subprocess.run('git clone --depth 1 https://github.com/anoneurx/astra.git %s' % REPO,
                   shell=True, check=True)
os.chdir(REPO)

# Checkpoints live outside the repo so a `git status` never sees them. Overridable so this
# notebook also runs outside Colab, where /content may not be writable.
RUNS = os.environ.get('ASTRA_RUNS', '/content/runs')
os.makedirs(RUNS, exist_ok=True)
DRIVE_ROOT = '/content/drive/MyDrive/astra/distillation_v2_salvaged'
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OK = os.path.isdir('/content/drive')
except Exception:
    DRIVE_OK = False
    print('Drive not available - results stay in /content and will be lost on reconnect')

assert os.path.exists('tools/salvage_distillation_v1.py'), \
    'repo is too old - salvage_distillation_v1.py is missing'
print('repo   :', os.getcwd())
print('python :', sys.version.split()[0])
print('drive  :', 'mounted' if DRIVE_OK else 'not mounted')
print('runs   :', RUNS)

## Stage 1 — Point at the V1 file

Drop `astra_v1_distillation.jsonl` into the panel, or set `V1_PATH` to wherever it lives.
The cell fails loudly rather than silently salvaging zero rows.

In [ ]:
# The env var is what makes this cell runnable outside Colab (CI, a plain Jupyter kernel).
# Leave it empty in Colab and the upload prompt handles it.
V1_PATH = os.environ.get('V1_PATH', '')

if not V1_PATH:
    if not os.path.exists('astra_v1_distillation.jsonl'):
        try:
            from google.colab import files
            print('Upload astra_v1_distillation.jsonl (1.1 MB, 1002 records)')
            files.upload()
        except ImportError:
            print('not running under Colab and the file is not here - set V1_PATH')
    V1_PATH = 'astra_v1_distillation.jsonl'

assert os.path.exists(V1_PATH), 'no such file: ' + V1_PATH
raw_n = sum(1 for l in open(V1_PATH, encoding='utf-8') if l.strip())
print('found %s: %d records, %.1f KiB'
      % (V1_PATH, raw_n, os.path.getsize(V1_PATH) / 1024))
assert raw_n > 0

## Stage 2 — Repair, dedup, split

One subprocess call into `tools/salvage_distillation_v1.py`, which is the same code as
`tools/` in the repo — so what you read here is what actually runs. It prints the per-reason
drop counts, the per-class survival table, the split sizes and the row-level overlap proof.

Defaults: input ≤ 480 chars, analysis ≤ 420, `normal`-class no-ops kept. Pass
`--max-input-chars 1200` to trade block-fit for ~220 more rows.

In [ ]:
MAX_INPUT_CHARS = 480
STRICT_NOOP = False      # True deletes ~97% of the normal class; see the header table

cmd = [sys.executable, 'tools/salvage_distillation_v1.py',
       '--in', V1_PATH, '--repo', REPO, '--force',
       '--max-input-chars', str(MAX_INPUT_CHARS)]
if STRICT_NOOP:
    cmd.append('--strict-noop')
print(' '.join(cmd), '\n')
subprocess.run(cmd, check=True)

## Stage 3 — Audit what landed

The same invariant set the v2 notebook enforces on generated data, so salvaged rows and
generated rows are held to one standard. `tests/test_salvage_parity.py` fails the build if
these two ever drift apart.

In [ ]:
from collections import Counter

CLASSIFICATIONS = ('normal', 'suspicious', 'malicious', 'unknown')
RISK_LEVELS = ('INFO', 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL')
HARDNESS_BY_RISK = {'INFO': 5, 'LOW': 20, 'MEDIUM': 45, 'HIGH': 70, 'CRITICAL': 90}
EARLY_WARNING_CLASSIFICATIONS = ('suspicious', 'malicious')
REQUIRED_FIELDS = ('input', 'classification', 'risk', 'early_warning',
                   'evidence', 'analysis', 'recommended_action', 'confidence')
KNOWN_FIELDS = frozenset(REQUIRED_FIELDS) | {
    'id', 'dedup_key', 'category', 'grid_index', 'source', 'synthetic',
    'teacher_model', 'created_at', 'hardness'}
MIN_INPUT_CHARS, MAX_INPUT_CHARS = 120, MAX_INPUT_CHARS
MIN_ANALYSIS_CHARS, MAX_ANALYSIS_CHARS = 40, 420
MIN_ACTION_CHARS, MAX_CONFIDENCE = 40, 0.97
REPEAT_ANCHOR, MAX_REPEATS = 60, 3
NO_OP_ACTIONS = ('no action required', 'no action needed', 'none', 'no action',
                 'no further action', 'nothing', 'n/a', 'not applicable')

DATA_DIR = 'datasets/distillation_v2'
RECORDS = os.path.join(DATA_DIR, 'records.jsonl')


def is_no_op(action):
    body = str(action).strip().rstrip('.!? ').lower()
    if body in NO_OP_ACTIONS:
        return True
    return any(body.startswith(p) and len(body) < len(p) + 90 for p in NO_OP_ACTIONS)


def is_repetitive(text, anchor=REPEAT_ANCHOR, max_repeats=MAX_REPEATS,
                  window=MAX_INPUT_CHARS):
    head = str(text)[:window]
    if len(head) < 4 * anchor:
        return False
    for off in (0, len(head) // 2, len(head) - 2 * anchor):
        frag = head[off:off + anchor]
        if len(frag) < anchor or not frag.strip():
            continue
        if head.count(frag) >= max_repeats:
            return True
    return False


records = [json.loads(l) for l in open(RECORDS, encoding='utf-8') if l.strip()]
print('auditing %d records\n' % len(records))
assert records, 'nothing was salvaged'

faults, accepted, keys = Counter(), Counter(), set()
for rec in records:
    if set(REQUIRED_FIELDS) - set(rec):
        faults['missing_field'] += 1
    for k in rec:
        if k not in KNOWN_FIELDS:
            faults['unknown_field'] += 1
    if '\n' in rec['input'] or '\n' in rec['analysis']:
        faults['multiline'] += 1
    if len(rec['input']) < MIN_INPUT_CHARS:
        faults['input_too_short'] += 1
    if len(rec['input']) > MAX_INPUT_CHARS:
        faults['input_too_long'] += 1
    if len(rec['analysis']) < MIN_ANALYSIS_CHARS:
        faults['analysis_too_short'] += 1
    if len(rec['analysis']) > MAX_ANALYSIS_CHARS:
        faults['analysis_too_long'] += 1
    if rec['classification'] not in CLASSIFICATIONS:
        faults['bad_classification'] += 1
    if rec['risk'] not in RISK_LEVELS:
        faults['bad_risk'] += 1
    if rec['hardness'] != HARDNESS_BY_RISK.get(rec['risk']):
        faults['hardness_mismatch'] += 1
    if is_repetitive(rec['input']) or is_repetitive(rec['analysis']):
        faults['repetitive'] += 1
    if is_no_op(rec['recommended_action']):
        if rec['classification'] in EARLY_WARNING_CLASSIFICATIONS:
            faults['action_is_no_op'] += 1
        elif rec['classification'] == 'normal':
            accepted['action_is_no_op_on_normal'] += 1
    if len(rec['recommended_action']) < MIN_ACTION_CHARS:
        faults['action_too_short'] += 1
    if bool(rec['early_warning']) != (rec['classification'] in EARLY_WARNING_CLASSIFICATIONS):
        faults['early_warning_inconsistent'] += 1
    if not (0.0 <= float(rec['confidence']) <= MAX_CONFIDENCE):
        faults['confidence_out_of_range'] += 1
    if rec['dedup_key'] in keys:
        faults['duplicate_key'] += 1
    keys.add(rec['dedup_key'])

print('invariant violations:')
if faults:
    for name, n in faults.most_common():
        print('  %-28s %5d  (%.2f%%)' % (name, n, 100.0 * n / len(records)))
    raise SystemExit('salvaged data violates the v2 contract - fix before training')
else:
    print('  none - every record satisfies every invariant')
if accepted:
    print('\naccepted deviations (reported, not faults):')
    for name, n in accepted.most_common():
        print('  %-28s %5d  (%.2f%%)' % (name, n, 100.0 * n / len(records)))

print('\nclassification :', dict(Counter(r['classification'] for r in records).most_common()))
print('risk           :', dict(Counter(r['risk'] for r in records).most_common()))
print('category       :', dict(Counter(r['category'] for r in records).most_common()))
print('mean input chars: %.0f' % (sum(len(r['input']) for r in records) / len(records)))
print('duplicate keys :', faults['duplicate_key'])

## Stage 4 — Rebuild the tokenizer over this corpus

This is not optional. The stock artifact is trained on prose + chat only, and it leaves
**16.02%** of the distilled security vocabulary as `<unk>` — words like `exfil-check` and
`serviceaccount` appear nowhere in the prose mix. `--extra` folds the salvaged corpora into the
*union* rather than replacing it, so the corpora the artifact was originally built for keep
their coverage (0.9893 → 0.9882), and the distilled text drops to 1.71% OOV.

In [ ]:
TOK_ARTIFACT = 'tokenizer/artifacts/prose_chat_word.json'
extras = [os.path.join(DATA_DIR, f, 'train.txt') for f in ('chat', 'alerts')]
extras = [p for p in extras if os.path.exists(p)]
assert extras, 'no salvaged corpora to fold in - rerun Stage 2'

cmd = [sys.executable, 'tools/build_word_tokenizer.py', '--extra', *extras,
       '--out', TOK_ARTIFACT]
print(' '.join(cmd), '\n')
subprocess.run(cmd, check=True)

## Stage 5 — Pre-flight: OOV, block length, leak gate

The student's block is 256 tokens, so an exchange that does not fit is split and the model
never sees the input and the answer in the same context.

Expect roughly this on a salvaged run, and read the numbers rather than the intent:

```
chat block length (student block = 256 tokens, 533 blocks)
  tokens min/p50/p90/p95/p99/max : 113 / 238 / 298 / 321 / 356 / 380
  over 256 tokens :   170 /   533  ( 31.9%)
  over 384 tokens :     0 /   533  (  0.0%)
```

**Lowering `MAX_INPUT_CHARS` will not fix this.** At the measured 3.17 chars/token, a 256-token
block holds about 812 characters for the input *and* the target together. Input ≤ 480 plus a
~500-char target is ~980, so it overflows. Clamping input alone still leaves 6.6% over 256 at
input ≤ 240, because the target is the binding side — to genuinely fit you have to shorten
`analysis` and `recommended_action`, not the telemetry.

The 480 cap is still worth having: it cut the worst-case block from V1's 421 tokens to 380, and
it rejects a runaway teacher at generation time instead of in the audit. And 31% is not
automatically wrong — the repo's own `datasets/chat` sits at 49.8% over 256, so the pipeline
already tolerates split exchanges.

In [ ]:
sys.path.insert(0, 'python')
from astra.tokenizer import load_tokenizer
from astra.safety import leak_check

BLOCK_TOKENS = 256
cfg = json.loads(open('configs/astra_distill_v2_salvaged_chat.json', encoding='utf-8').read())
tok = load_tokenizer(cfg['tokenizer'])
unk = tok.special_names.index('<unk>')
print('tokenizer: %d tokens (%d words), unk_id=%d\n'
      % (len(tok), tok.stats.get('num_words', 0), unk))

texts = {}
for flavour in ('chat', 'alerts'):
    for split in ('train', 'val'):
        path = os.path.join(DATA_DIR, flavour, split + '.txt')
        assert os.path.exists(path), 'missing corpus: ' + path
        texts['%s/%s' % (flavour, split)] = open(path, encoding='utf-8').read()

for name, text in texts.items():
    ids = tok.encode(text)
    oov = sum(1 for i in ids if i == unk)
    print('%-14s %8d chars %8d tokens  OOV=%5.2f%%  blocks=%4d'
          % (name, len(text), len(ids), 100.0 * oov / max(1, len(ids)),
             len(text) // BLOCK_TOKENS))

blocks = [b for b in texts['chat/train'].split('\n\n') if b.strip()]
blen = sorted(len(tok.encode(b)) for b in blocks)
n = len(blen)
print('\nchat block length (student block = %d tokens, %d blocks)' % (BLOCK_TOKENS, n))
print('  tokens min/p50/p90/p95/p99/max : %d / %d / %d / %d / %d / %d'
      % (blen[0], blen[n // 2], blen[int(n * .90)], blen[int(n * .95)],
         blen[int(n * .99)], blen[-1]))
for lim in (256, 384, 512):
    over = sum(1 for x in blen if x > lim)
    print('  over %3d tokens : %5d / %5d  (%5.1f%%)'
          % (lim, over, n, 100.0 * over / max(1, n)))

result = leak_check(tok.encode(texts['chat/train']),
                    tok.encode(texts['chat/val']), n=13)
print('\nleak_check(chat train, chat val, n=13) =')
print(json.dumps(result, indent=2))
print('\nThat is the number the disabled gate would have tripped on. It is off because a repeated')
print('chat framing template makes any overlap meaningless here; the record-level proof lives')
print('in manifest.json under row_level_safety.')

manifest_path = os.path.join(DATA_DIR, 'manifest.json')
manifest = json.loads(open(manifest_path, encoding='utf-8').read())
assert all(v == 0 for v in manifest['row_level_safety'].values()), 'a record is in two splits'
manifest['preflight'] = {
    'tokenizer': cfg['tokenizer'],
    'tokenizer_vocab_size': len(tok),
    'block_tokens': BLOCK_TOKENS,
    'chat_block_tokens': {
        'count': n, 'min': blen[0], 'p50': blen[n // 2], 'p90': blen[int(n * .90)],
        'p95': blen[int(n * .95)], 'p99': blen[int(n * .99)], 'max': blen[-1],
        'over_256': sum(1 for x in blen if x > 256),
    },
    'leak_report': result,
}
with open(manifest_path, 'w', encoding='utf-8') as fh:
    json.dump(manifest, fh, indent=2, sort_keys=True)
    fh.write('\n')
print('\nleak_report + preflight recorded in', manifest_path)

print('\nfirst train block:')
print(blocks[0][:600])

## Stage 6 — Inspect the emitted files

Worth eyeballing before spending GPU hours. If the first block reads like garbage the whole
corpus will.

In [ ]:
for rel in ('records.jsonl', 'manifest.json',
            'chat/train.txt', 'chat/val.txt', 'alerts/train.txt'):
    path = os.path.join(DATA_DIR, rel)
    if os.path.exists(path):
        print('  %-24s %9d bytes' % (rel, os.path.getsize(path)))

print('\nfirst chat train block:')
print(open(os.path.join(DATA_DIR, 'chat/train.txt'), encoding='utf-8').read().split('\n\n')[0])
print('\nfirst alerts train row:')
print(open(os.path.join(DATA_DIR, 'alerts/train.txt'), encoding='utf-8').readline().strip())

for rel in ('configs/astra_distill_v2_salvaged_chat.json',
            'configs/astra_distill_v2_salvaged_alerts.json'):
    assert os.path.exists(rel), 'missing config: ' + rel
print('\nconfigs written for both corpus flavours')

## Stage 7 — Mirror to Drive

`datasets/*/*.jsonl` is gitignored (`.gitignore:17`), which is exactly why V1 was unversionable.
The `.txt` splits and `manifest.json` are not ignored — those are the trackable artifacts. The
tokenizer artifact is a build output and is ignored too, so rebuild it on a fresh clone.

In [ ]:
if not DRIVE_OK:
    print('Drive not mounted - skipping. Results are in /content and will be lost on reconnect.')
else:
    dst = os.path.join(DRIVE_ROOT, 'datasets', DATA_DIR)
    os.makedirs(dst, exist_ok=True)
    copied = []
    for root, _dirs, files in os.walk(DATA_DIR):
        for name in files:
            src = os.path.join(root, name)
            out = os.path.join(dst, os.path.relpath(src, DATA_DIR))
            os.makedirs(os.path.dirname(out), exist_ok=True)
            shutil.copy2(src, out)
            copied.append(os.path.relpath(src, DATA_DIR))
    for rel in ('configs/astra_distill_v2_salvaged_chat.json',
                'configs/astra_distill_v2_salvaged_alerts.json'):
        out = os.path.join(DRIVE_ROOT, os.path.basename(rel))
        shutil.copy2(rel, out)
        copied.append(os.path.basename(rel))
    print('mirrored %d files to %s' % (len(copied), DRIVE_ROOT))
    for c in sorted(copied):
        print('  ', c)

## Stage 8 — Training

This is the cell that turns the corpus into a model, so it is **on**. Two stages, mirroring
the v2 notebook: **alerts first** (sequence + label head), then **chat warm-started from the
alerts checkpoint** with `--reset-step` so the LR schedule and step counter restart rather
than resuming mid-schedule.

`gpu_train.py:380` writes `<out>/final.npz`, or `<out>/resumed/final.npz` when `--resume` is
passed. `find_final` globs for both rather than assuming one layout.

**Set the Colab runtime to a GPU first** (Runtime → Change runtime type → T4). The cell
asserts on `cuda.is_available()` rather than falling back to CPU, because a CPU run of 3800
steps on this model does not finish in any reasonable time and would look like a hang.

`bf16` is the requested dtype and is used whenever torch reports the GPU as supporting it.
On a **T4 (sm_75)** current torch still reports `is_bf16_supported() == True`, because it
emulates bf16 there, so bf16 is what a T4 run uses. The `is_bf16_supported()` check is kept
for older torch and for GPUs that genuinely lack it, where `torch.amp.autocast` raises
`Current CUDA Device does not support bfloat16`. No GradScaler is needed on the fp16 path
because `gpu_train.py` does not scale the loss.

Two things to expect from a corpus this size, so the numbers are not a surprise later:

* 129k chat tokens is ~505 blocks of 256. At 3800 steps x batch 8 that is roughly **60
  epochs** over 533 rows, so this will memorise the training split. It is a working
  pipeline and a real checkpoint, not a converged model.
* Validation is thin: 1534 chat val tokens is **6 blocks**, 749 alert val tokens is under
  3. Treat val loss as a smoke signal, not a selection signal.

In [ ]:
RUN_TRAIN = True
DTYPE = 'bf16'


def run_logged(cmd, log):
    """Run a training command, streaming its output live but keeping a log.

    `subprocess.run(shell=True, check=True)` is what hid the real error the first time this
    ran: the CalledProcessError carried the command and not the message, so the actual
    failure only existed in a scrollback nobody read. Stream it AND keep the tail."""
    print('\n$ ' + cmd, flush=True)
    with open(log, 'w', encoding='utf-8') as fh:
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            sys.stdout.write(line)
            fh.write(line)
    rc = proc.wait()
    if rc:
        tail = ''.join(open(log, encoding='utf-8').readlines()[-40:])
        raise SystemExit('%s\n\nfailed with exit %d. Last 40 lines of %s:\n%s'
                         % (cmd, rc, log, tail))


def find_final(out_dir):
    plain = os.path.join(out_dir, 'final.npz')
    if os.path.exists(plain):
        return plain
    hits = sorted(glob.glob(os.path.join(out_dir, '**', 'final.npz'), recursive=True))
    return hits[0] if hits else None


def persist(out_dir):
    if not DRIVE_OK:
        return
    dst = os.path.join(DRIVE_ROOT, 'runs', os.path.basename(out_dir))
    os.makedirs(dst, exist_ok=True)
    for src in [os.path.join(out_dir, n) for n in ('final.npz', 'report.json')] + \
            glob.glob(os.path.join(out_dir, '*.manifest.json')):
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(dst, os.path.basename(src)))


if not RUN_TRAIN:
    print('RUN_TRAIN is False - set it to True to spend GPU time.')
else:
    import torch
    assert torch.cuda.is_available(), (
        'No GPU visible. Runtime -> Change runtime type -> T4, then re-run this cell. '
        'Falling back to CPU here would look like a hang rather than an error.')
    gpu = torch.cuda.get_device_name(0)
    print('torch %s  cuda=%s  %s' % (torch.__version__, torch.cuda.is_available(), gpu))
    if DTYPE == 'bf16' and not torch.cuda.is_bf16_supported():
        print('%s has no native bfloat16 (sm_%d) - torch.amp.autocast raises on it, '
              'so using fp16' % (gpu, torch.cuda.get_device_capability(0)[0]
                                 * 10 + torch.cuda.get_device_capability(0)[1]))
        DTYPE = 'fp16'
    print('dtype:', DTYPE)

    out_alerts = os.path.join(RUNS, 'salvage_alerts')
    os.makedirs(out_alerts, exist_ok=True)
    run_logged('%s training/gpu_train.py '
               '--config configs/astra_distill_v2_salvaged_alerts.json '
               '--out %s --cache-dir /tmp/astra_cache --dtype %s'
               % (sys.executable, out_alerts, DTYPE), 'train_alerts.log')
    alerts_final = find_final(out_alerts)
    assert alerts_final, 'alerts stage produced no final.npz'
    print('alerts final:', alerts_final)
    persist(out_alerts)

    assert os.path.exists(alerts_final), 'alerts final.npz vanished - rerun stage 1'
    out_chat = os.path.join(RUNS, 'salvage_chat')
    os.makedirs(out_chat, exist_ok=True)
    run_logged('%s training/gpu_train.py '
               '--config configs/astra_distill_v2_salvaged_chat.json '
               '--out %s --resume %s --reset-step '
               '--cache-dir /tmp/astra_cache --dtype %s'
               % (sys.executable, out_chat, alerts_final, DTYPE), 'train_chat.log')
    chat_final = find_final(out_chat)
    assert chat_final, 'chat stage produced no final.npz'
    print('chat final:', chat_final)
    persist(out_chat)
    print('\ndone - checkpoints under %s'
          % (DRIVE_ROOT if DRIVE_OK else '(no Drive, so only on this VM)'))

## Stage 9 — Smoke test

A checkpoint that loads is not a checkpoint that learned something. This runs the distilled
chat model on three prompts and prints a held-out alert row, so you can see the output shape
before trusting it. Expect weak or garbled text - see the Stage 8 note on epochs. If the
reply comes back empty or as special tokens, the tokenizer and the checkpoint disagree and
that is a real bug, not a quality problem.

In [ ]:
import numpy as np
from astra.inference import KVCache, decode
from astra.learning.evaluate import load_model
from astra.model import ModelConfig
from astra.tokenizer.word import WordLevel

chat_final = find_final(os.path.join(RUNS, 'salvage_chat'))
assert chat_final, 'no chat checkpoint - run Stage 8 first'
print('checkpoint:', chat_final)

cfg = json.load(open('configs/astra_distill_v2_salvaged_chat.json', encoding='utf-8'))
tok = WordLevel.load(cfg['tokenizer'])
mcfg = ModelConfig.from_dict(cfg['model'])
mcfg.vocab_size = len(tok)

model = load_model(mcfg, chat_final)
cache = KVCache(mcfg)
SPECIALS = set(range(tok.num_special))
history = []


def ask(user_text, max_new=64, temperature=0.7, top_k=40, rep_penalty=1.15):
    global history
    history.append('You: %s' % user_text)
    prompt = '\\n'.join(history) + '\\nAstra:'
    ids = tok.encode(prompt)[-(mcfg.max_seq_len - max_new):]
    new = decode(model, ids, max_new=max_new, temperature=temperature,
                 rng=np.random.default_rng(len(history)), top_k=top_k,
                 cache=cache, rep_penalty=rep_penalty, forbidden=SPECIALS)
    reply = tok.decode(new).split('\\nYou:')[0].strip()
    history.append('Astra: %s' % reply)
    assert reply, 'empty reply - the checkpoint and tokenizer disagree'
    return reply


for q in ['an ssh login as root from 203.0.113.9 at 02:14 outside the change window',
          'explain what early warning means in a security alert stream',
          'what should an analyst do about a service account whose permissions widened']:
    print('You  :', q)
    print('Astra:', ask(q))
    print()

alert_path = os.path.join(DATA_DIR, 'alerts/eval.txt')
if os.path.exists(alert_path):
    print('held-out alert row (never trained on):')
    print(' ', open(alert_path, encoding='utf-8').readline().strip()[:320])

## Stage 10 — Download

One zip, because the pieces are useless apart. The one people forget is the **tokenizer**:
`.gitignore:19` excludes `tokenizer/artifacts/`, so it is not in the repo and cannot be
re-cloned. Every config points at it, so without it nothing loads.

* `datasets/distillation_v2/**` — the corpora `gpu_train.py` reads, plus `records.jsonl`
  (provenance: `id`, `dedup_key`, `grid_index`, `teacher_model`) and `manifest.json`
  (every drop count, every repair, the leak report and preflight).
* `configs/…salvaged_{chat,alerts}.json` — paths already point at the data above.
* `tokenizer/artifacts/prose_chat_word.json` — **required**, rebuilt with `--extra`.
* `runs/**/final.npz` — the checkpoints, only if you set `RUN_TRAIN = True`. This is the
  actual model; everything else is inputs to it.

`SHA256SUMS.txt` is included so you can check the zip arrived intact.

In [ ]:
import zipfile, hashlib

want = []
want += sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.txt'), recursive=True))
want += [os.path.join(DATA_DIR, 'records.jsonl'), os.path.join(DATA_DIR, 'manifest.json')]
want += ['configs/astra_distill_v2_salvaged_chat.json',
         'configs/astra_distill_v2_salvaged_alerts.json']
want += [TOK_ARTIFACT]
want += sorted(glob.glob(os.path.join(RUNS, '**', 'final.npz'), recursive=True))
want += sorted(glob.glob(os.path.join(RUNS, '**', 'report.json'), recursive=True))

have = [p for p in want if os.path.exists(p)]
skipped = [os.path.basename(p) for p in want if not os.path.exists(p)]
if skipped:
    print('not produced, skipping:', skipped)
assert have, 'nothing to package - did Stage 2 run?'


def arcname(path):
    """The name this file has INSIDE the zip. Checkpoints sit under RUNS, which is outside
    the repo, so they cannot be laid out relative to the repo without climbing out with '..'
    - and SHA256SUMS.txt has to list exactly these names or `sha256sum -c` cannot verify."""
    if os.path.abspath(path).startswith(os.path.abspath(RUNS) + os.sep):
        return os.path.join('runs', os.path.relpath(path, RUNS))
    return os.path.relpath(path, REPO)


arcs = {p: arcname(p) for p in have}

# relative to the repo dir rather than /content, so this also runs outside Colab
OUT = os.path.abspath('astra_distill_v2_salvaged.zip')
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in have:
        z.write(p, arcs[p])
    z.writestr('SHA256SUMS.txt', '\n'.join(
        '%s  %s' % (hashlib.sha256(open(p, 'rb').read()).hexdigest(), arcs[p])
        for p in have) + '\n')

print('%-52s %10s' % ('file', 'bytes'))
for p in have:
    print('%-52s %10d' % (arcs[p], os.path.getsize(p)))
print('\n%d files, %.1f MB zipped' % (len(have), os.path.getsize(OUT) / 1e6))

if not DRIVE_OK:
    try:
        from google.colab import files
        print('\ndownloading', OUT)
        files.download(OUT)
    except ImportError:
        print('\nnot under Colab - the zip is at', OUT)
else:
    shutil.copy2(OUT, os.path.join(DRIVE_ROOT, os.path.basename(OUT)))
    print('\ncopied to Drive:', os.path.join(DRIVE_ROOT, os.path.basename(OUT)))